# Backpropagation

<img src="img/title.png" width="600">

<br><br><br>

## Why we need derivatives

We want to minimize a loss function in terms of some model parameters $\ell(p_1, p_2, \ldots, p_{N_p})$.

In general, we'll need a non-linear optimization algorithm.

Some optimization algorithms work with just the function:
* Nelder-Mead simplex
* Powell's method
* Simulated annealing (inspired by cooling metal)
* Genetic algorithms (inspired by biological evolution)
* ...

But algorithms that are given the derivative,

$$ \nabla \ell = \left( \frac{d\ell}{dp_1}, \frac{d\ell}{dp_2}, \ldots, \frac{d\ell}{dp_{N_p}} \right) $$

need to evaluate fewer points $(p_1, p_2, \ldots, p_{N_p})$ to determine the shape of the surface and therefore find the minimum faster.

* Gradient descent
* BFGS, L-BFGS-B
* Adam
* Stochastic gradient descent
* ...

<br><br><br>

<img src="img/downhill.png" width="600">

<br><br><br>

## How computers compute derivatives

* Symbolic: like doing calculus by hand
* Finite differences: numerical with a "small enough" step size $\Delta p_i$
* Autodiff: by propagating each operation using the chain rule

<br><br><br>

## Example of something to differentiate

<img src="img/example.png" width="600">

Activation function:

$$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

First layer:

$$ \begin{aligned}
z_1 &= p_1 \, x_1 + p_4 \, x_2 + p_7 \\
z_2 &= p_2 \, x_1 + p_5 \, x_2 + p_8 \\
z_3 &= p_3 \, x_1 + p_6 \, x_2 + p_9 \\
\end{aligned} $$

Second layer:

$$ \hat{y} = p_{10} \, \sigma(z_1) + p_{11} \, \sigma(z_2) + p_{12} \, \sigma(z_3) + p_{13} $$

Loss is predicted minus target squared:

$$ \ell(p_1, p_2, \ldots p_{13}) = \left( \hat{y} - y \right)^2 $$

<br><br><br>

## Symbolic differentiation

In [11]:
import sympy as sp

# Symbols
x1, x2, y = sp.symbols("x1 x2 y")
(p1, p2, p3, p4, p5, p6, p7, p8, p9,
 p10, p11, p12, p13) = params = sp.symbols("p1:14")

# Activation function
def sigma(z):
    return 1 / (1 + sp.exp(-z))

# First layer
z1 = p1*x1 + p4*x2 + p7
z2 = p2*x1 + p5*x2 + p8
z3 = p3*x1 + p6*x2 + p9

# Second layer
yhat = p10*sigma(z1) + p11*sigma(z2) + p12*sigma(z3) + p13

# Loss
ell = (yhat - y)**2

# Derivatives d ell / d p_i
derivatives = {
    p_i: sp.simplify(sp.diff(ell, p_i))
    for p_i in params
}

In [21]:
# Print results
sp.init_printing(num_columns=10000)
for p_i, deriv in derivatives.items():
    print()
    print(f"dℓ / d{p_i} =")
    sp.pprint(deriv)


dℓ / dp1 =
         ⎛          p₁₀                        p₁₁                        p₁₂                     ⎞  -p₁⋅x₁ - p₄⋅x₂ - p₇
2⋅p₁₀⋅x₁⋅⎜──────────────────────── + ──────────────────────── + ──────────────────────── + p₁₃ - y⎟⋅ℯ                   
         ⎜ -p₁⋅x₁ - p₄⋅x₂ - p₇        -p₂⋅x₁ - p₅⋅x₂ - p₈        -p₃⋅x₁ - p₆⋅x₂ - p₉              ⎟                     
         ⎝ℯ                    + 1   ℯ                    + 1   ℯ                    + 1          ⎠                     
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                        2                                               
                                              ⎛ -p₁⋅x₁ - p₄⋅x₂ - p₇    ⎞                                                
                                              ⎝ℯ                    + 1⎠                                                

dℓ / dp2 =
        

<br><br><br>

In [24]:
import random

values = {
    x1: random.uniform(-1, 1),
    x2: random.uniform(-1, 1),
    y: random.uniform(-1, 1),
}
for p_i in params:
    values[p_i] = random.uniform(-1, 1)

print(f"ℓ = {ell.subs(values).evalf():.2f}\n")

for p_i, deriv in derivatives.items():
    print(f"dℓ / d{p_i} = {deriv.subs(values).evalf():.2f}")

ℓ = 0.15

dℓ / dp1 = -0.16
dℓ / dp2 = 0.05
dℓ / dp3 = 0.02
dℓ / dp4 = 0.06
dℓ / dp5 = -0.02
dℓ / dp6 = -0.01
dℓ / dp7 = 0.18
dℓ / dp8 = -0.06
dℓ / dp9 = -0.02
dℓ / dp10 = 0.36
dℓ / dp11 = 0.33
dℓ / dp12 = 0.32
dℓ / dp13 = 0.77

<br><br><br>

This does not scale well to a large number of parameters.

<br><br><br>